# 08 — Evaluasi Sistem Terintegrasi (YOLOv12 + CNN Terbaik)

Notebook ini menjalankan **evaluasi tambahan** (Tabel 4.2) untuk membuktikan sistem
bekerja secara utuh: YOLOv12 mendeteksi biji pada foto multi-bean, hasil deteksi
di-crop, lalu CNN terbaik menilai kualitas tiap crop.

Bedanya dengan evaluasi utama (notebook 07 / Tabel 4.1):

| Aspek | Evaluasi utama (Tabel 4.1) | Evaluasi terintegrasi (Tabel 4.2) |
| --- | --- | --- |
| Sumber crop | koordinat anotasi ground truth | prediksi bounding box YOLOv12 |
| Tujuan | perbandingan arsitektur CNN yang adil | bukti sistem berfungsi end-to-end |
| Error segmentasi | tidak ikut terhitung | ikut terhitung (FP/FN YOLOv12) |

Prediksi YOLOv12 dicocokkan ke anotasi ground truth via **IoU greedy matching**
(`src/evaluation/matching.py`) agar label kualitas tiap deteksi diketahui.

In [ ]:
# Bootstrap: tambahkan root proyek ke sys.path agar modul src/ & configs/ dapat diimpor.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from PIL import Image

from configs.base_config import (
    CLASS_NAMES,
    DATA_DIR,
    DEVICE,
    END_TO_END_RESULTS_DIR,
    IOU_MATCH_THRESHOLD,
    METRICS_DIR,
    MODELS_DIR,
    RANDOM_SEED,
)
from src.evaluation.confusion_matrix import plot_confusion_matrix
from src.evaluation.metrics import compute_metrics
from src.inference.end_to_end import EndToEndPipeline
from src.utils.seed import set_seed

In [ ]:
# Reproducibility & device.
set_seed(RANDOM_SEED)
print('device :', DEVICE)
print('kelas  :', CLASS_NAMES)
print('iou_thr:', IOU_MATCH_THRESHOLD)

In [ ]:
# Bobot YOLOv12 terlatih (hasil notebook 03; salin best.pt dari runs/ ke results/models/).
YOLO_WEIGHTS = MODELS_DIR / 'yolov12_best.pt'
print(YOLO_WEIGHTS, '->', 'ADA' if YOLO_WEIGHTS.exists() else 'BELUM ADA')

In [ ]:
# CNN terbaik — ganti CNN_ARCH & CNN_WEIGHTS sesuai pemenang Tabel 4.1.
CNN_ARCH = 'resnet50'  # {'resnet50', 'mobilenetv3', 'inceptionv3'}

candidates = sorted((MODELS_DIR).glob(f'**/{CNN_ARCH}_best_*.pt'))
CNN_WEIGHTS = candidates[-1] if candidates else MODELS_DIR / f'{CNN_ARCH}_best.pt'
print('checkpoint:', CNN_WEIGHTS, '->', 'ADA' if CNN_WEIGHTS.exists() else 'BELUM ADA')

In [ ]:
pipeline = EndToEndPipeline(
    yolo_weights=YOLO_WEIGHTS,
    cnn_weights=CNN_WEIGHTS,
    cnn_arch=CNN_ARCH,
    device=DEVICE,
    iou_threshold=IOU_MATCH_THRESHOLD,
)
pipeline

## Jalankan Evaluasi

Seluruh foto pada split test diproses: deteksi → matching IoU → crop → klasifikasi.
Hasil lengkap disimpan ke `results/end_to_end/end_to_end_results.json`.

In [ ]:
TEST_IMAGES = DATA_DIR / 'annotated' / 'test' / 'images'
TEST_LABELS = DATA_DIR / 'annotated' / 'test' / 'labels'

results = pipeline.evaluate_on_test_set(
    test_images_dir=TEST_IMAGES,
    test_labels_dir=TEST_LABELS,
    output_dir=END_TO_END_RESULTS_DIR,
)

metrics = results['metrics']
pd.DataFrame([metrics]).T.rename(columns={0: 'nilai'})

In [ ]:
# Confusion matrix CNN pada crop hasil deteksi YOLOv12.
if results['y_true']:
    fig = plot_confusion_matrix(
        results['y_true'],
        results['y_pred'],
        class_names=CLASS_NAMES,
        title=f'Confusion Matrix End-to-End — {CNN_ARCH}',
        output_path=END_TO_END_RESULTS_DIR / f'{CNN_ARCH}_e2e_confusion_matrix.png',
    )
    display(fig)
else:
    print('Belum ada matched pair — periksa bobot YOLOv12 atau turunkan iou_threshold.')

## Visualisasi Contoh

Tiga foto test dipilih acak (seed tetap). Bounding box hasil YOLOv12 di-overlay
dengan label CNN: **hijau = layak**, **merah = tidak layak**.

In [ ]:
COLORS = {'layak': 'lime', 'tidak_layak': 'red'}

image_paths = sorted(p for p in TEST_IMAGES.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
random.seed(RANDOM_SEED)
samples = random.sample(image_paths, k=min(3, len(image_paths)))

for image_path in samples:
    out = pipeline.predict_single_image(image_path)
    image = Image.open(image_path).convert('RGB')

    fig, ax = plt.subplots(figsize=(11, 8))
    ax.imshow(image)
    ax.axis('off')

    for bbox, pred, probs in zip(out['bboxes'], out['cnn_predictions'], out['cnn_probabilities']):
        x1, y1, x2, y2 = bbox
        label = CLASS_NAMES[pred]
        color = COLORS.get(label, 'yellow')
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2))
        ax.text(
            x1, max(y1 - 4, 0), f'{label} {probs[pred]:.2f}',
            color='black', fontsize=7,
            bbox={'facecolor': color, 'alpha': 0.75, 'pad': 1, 'edgecolor': 'none'},
        )

    ax.set_title(f"{image_path.name} — {out['num_detections']} biji terdeteksi")
    plt.tight_layout()
    plt.show()

## Perbandingan dengan Tabel 4.1

Metrik CNN pada crop **ground truth** (evaluasi utama, `results/metrics/`) versus
pada crop **hasil deteksi YOLOv12** (evaluasi terintegrasi). Selisihnya adalah biaya
yang ditimbulkan tahap segmentasi.

In [ ]:
gt_metrics_path = METRICS_DIR / f'{CNN_ARCH}_test_metrics.json'

if gt_metrics_path.exists():
    gt_metrics = json.loads(gt_metrics_path.read_text(encoding='utf-8'))
    comparison = pd.DataFrame(
        {
            'Crop ground truth (Tabel 4.1)': [
                gt_metrics['accuracy'], gt_metrics['precision'], gt_metrics['recall'], gt_metrics['f1'],
            ],
            'Crop YOLOv12 (Tabel 4.2)': [
                metrics['cnn_accuracy'], metrics['cnn_precision'], metrics['cnn_recall'], metrics['cnn_f1'],
            ],
        },
        index=['accuracy', 'precision', 'recall', 'f1'],
    )
    comparison['selisih'] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
    display(comparison.round(4))
else:
    print(f'Belum ada {gt_metrics_path.name}. Jalankan: python -m scripts.evaluate --model {CNN_ARCH} --checkpoint <path>')

## Analisis Matching

Statistik pencocokan per foto menunjukkan seberapa banyak biji yang berhasil
dilokalisasi YOLOv12 (**TP**), terdeteksi tetapi tidak ada di anotasi (**FP**),
dan terlewat sama sekali (**FN**).

In [ ]:
per_image = pd.DataFrame(results['per_image'])

if not per_image.empty:
    ax = per_image.set_index('image')[['true_positives', 'false_positives', 'false_negatives']].plot(
        kind='bar', stacked=False, figsize=(13, 5),
        color=['tab:green', 'tab:orange', 'tab:red'],
    )
    ax.set_ylabel('jumlah biji')
    ax.set_xlabel('foto test')
    ax.set_title(f'Statistik Matching YOLOv12 per Foto (IoU >= {IOU_MATCH_THRESHOLD})')
    plt.xticks(rotation=90, fontsize=7)
    plt.tight_layout()
    plt.show()

    display(per_image[['true_positives', 'false_positives', 'false_negatives']].describe().round(2))

    print(
        f"\nRingkasan: {metrics['num_matched']} biji ter-match, "
        f"{metrics['num_false_positives']} FP, {metrics['num_false_negatives']} FN.\n"
        f"YOLO precision={metrics['yolo_precision']:.4f}, recall={metrics['yolo_recall']:.4f}.\n"
        'FN tinggi → biji terlewat (biji saling menempel / terlalu kecil); '
        'FP tinggi → deteksi ganda atau latar terbaca sebagai biji. '
        'Keduanya tidak masuk metrik CNN, tetapi menurunkan kegunaan sistem di lapangan.'
    )
else:
    print('Belum ada hasil per-foto.')